In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 07_train_val_test_split
# MAGIC Split 70/15/15 para train/val/test (seed=42)

# COMMAND ----------

from sklearn.model_selection import train_test_split
import pandas as pd

GOLD_PATH = "/Volumes/olist/olist_gold/gold/"
SPLIT_PATH = "/Volumes/olist/olist_gold/model_split/"

print("🚀 Iniciando split train/val/test\n")

# COMMAND ----------

# Crear volumes si no existen
try:
    spark.sql("CREATE VOLUME IF NOT EXISTS olist.olist_gold.model_split")
    print("✅ Volume model_split verificado\n")
except:
    pass

# COMMAND ----------

# Cargar dataset reducido (PCA)
print("📥 Cargando dataset reducido...\n")

df = spark.read.format("delta").load(f"{GOLD_PATH}customer_features_rfm_20180930_reduced/").toPandas()

print(f"✅ {len(df):,} registros, {len(df.columns)} columnas\n")

# COMMAND ----------

# Separar X y y
print("🎯 Separando features y target...\n")

# Excluir customer_id (si existe) y target
X_cols = [c for c in df.columns if c not in ["customer_id", "is_premium"]]
X = df[X_cols]
y = df["is_premium"]

print(f"✅ Features (X): {len(X_cols)} columnas")
print(f"✅ Target (y): is_premium")
print(f"✅ Distribución target: {y.value_counts().to_dict()}\n")

# COMMAND ----------

# Split 70/15/15  ##################################################################
print("✂️  Split train/val/test (70/15/15)...\n")

# Train 70%, temp 30%
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Val 15%, Test 15% (50/50 del temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"✅ Train: {len(X_train):,} ({len(X_train)/len(df)*100:.1f}%)")
print(f"✅ Val:   {len(X_val):,} ({len(X_val)/len(df)*100:.1f}%)")
print(f"✅ Test:  {len(X_test):,} ({len(X_test)/len(df)*100:.1f}%)\n")

# COMMAND ----------

# Crear DataFrames completos
print("📦 Creando DataFrames...\n")

train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

print(f"✅ Train: {train_df.shape}")
print(f"✅ Val:   {val_df.shape}")
print(f"✅ Test:  {test_df.shape}\n")

# COMMAND ----------

# Guardar como Delta
print("💾 Guardando tablas Delta...\n")

spark.createDataFrame(train_df).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{SPLIT_PATH}train/")
print("✅ Train guardado")

spark.createDataFrame(val_df).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{SPLIT_PATH}val/")
print("✅ Val guardado")

spark.createDataFrame(test_df).write \
    .format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(f"{SPLIT_PATH}test/")
print("✅ Test guardado\n")

# COMMAND ----------

# Guardar como CSV
print("💾 Guardando CSV...\n")

# Crear directorio csv
dbutils.fs.mkdirs(f"{SPLIT_PATH}csv/")

# Train CSV
spark.createDataFrame(train_df).coalesce(1).write \
    .format("csv").mode("overwrite").option("header", "true") \
    .save(f"{SPLIT_PATH}csv/train_temp/")
csv = [f for f in dbutils.fs.ls(f"{SPLIT_PATH}csv/train_temp/") if f.name.endswith('.csv')][0]
dbutils.fs.cp(csv.path, f"{SPLIT_PATH}csv/train.csv")
dbutils.fs.rm(f"{SPLIT_PATH}csv/train_temp/", True)
print("✅ train.csv")

# Val CSV
spark.createDataFrame(val_df).coalesce(1).write \
    .format("csv").mode("overwrite").option("header", "true") \
    .save(f"{SPLIT_PATH}csv/val_temp/")
csv = [f for f in dbutils.fs.ls(f"{SPLIT_PATH}csv/val_temp/") if f.name.endswith('.csv')][0]
dbutils.fs.cp(csv.path, f"{SPLIT_PATH}csv/val.csv")
dbutils.fs.rm(f"{SPLIT_PATH}csv/val_temp/", True)
print("✅ val.csv")

# Test CSV
spark.createDataFrame(test_df).coalesce(1).write \
    .format("csv").mode("overwrite").option("header", "true") \
    .save(f"{SPLIT_PATH}csv/test_temp/")
csv = [f for f in dbutils.fs.ls(f"{SPLIT_PATH}csv/test_temp/") if f.name.endswith('.csv')][0]
dbutils.fs.cp(csv.path, f"{SPLIT_PATH}csv/test.csv")
dbutils.fs.rm(f"{SPLIT_PATH}csv/test_temp/", True)
print("✅ test.csv\n")

# COMMAND ----------

# Verificar distribución de target en cada split
print("📊 Distribución del target por split:\n")

print("Train:")
print(train_df['is_premium'].value_counts(normalize=True).round(3))
print(f"\nVal:")
print(val_df['is_premium'].value_counts(normalize=True).round(3))
print(f"\nTest:")
print(test_df['is_premium'].value_counts(normalize=True).round(3))
print()

# COMMAND ----------

# Resumen final
print(f"{'='*60}")
print("✅ SPLIT COMPLETADO")
print(f"{'='*60}")
print(f"Total registros: {len(df):,}")
print(f"Train: {len(train_df):,} (70%)")
print(f"Val:   {len(val_df):,} (15%)")
print(f"Test:  {len(test_df):,} (15%)")
print(f"\nOutputs Delta:")
print(f"  • {SPLIT_PATH}train/")
print(f"  • {SPLIT_PATH}val/")
print(f"  • {SPLIT_PATH}test/")
print(f"\nOutputs CSV:")
print(f"  • {SPLIT_PATH}csv/train.csv")
print(f"  • {SPLIT_PATH}csv/val.csv")
print(f"  • {SPLIT_PATH}csv/test.csv")